# Task 2: Power BI Star-Schema Dimensional Modeling & Preparation
**Framework**: RISK (Requirements & Research, Investigation & Insight Discovery, Solution Engineering, Knowledge Delivery & KPI Validation)  
**Objective**: Build and validate an enterprise-grade Star Schema (`FactSales`, `DimDate`, `DimCustomer`, `DimProduct`, `DimGeography`), verify 100% referential integrity, and simulate DAX time-intelligence measures in Python for reconciliation.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

STAR_DIR = PROJECT_ROOT / "data" / "processed" / "star_schema"

# Load Star Schema Tables
fact_sales = pd.read_csv(STAR_DIR / "FactSales.csv", dtype={'Postal Code': str})
dim_date = pd.read_csv(STAR_DIR / "DimDate.csv")
dim_customer = pd.read_csv(STAR_DIR / "DimCustomer.csv")
dim_product = pd.read_csv(STAR_DIR / "DimProduct.csv")
dim_geography = pd.read_csv(STAR_DIR / "DimGeography.csv", dtype={'Postal Code': str})

print("Star-Schema Tables Loaded Successfully:")
print(f"FactSales: {fact_sales.shape}")
print(f"DimDate: {dim_date.shape}")
print(f"DimCustomer: {dim_customer.shape}")
print(f"DimProduct: {dim_product.shape}")
print(f"DimGeography: {dim_geography.shape}")

Star-Schema Tables Loaded Successfully:
FactSales: (9994, 16)
DimDate: (1826, 12)
DimCustomer: (793, 3)
DimProduct: (1862, 4)
DimGeography: (631, 5)


## 1. Referential Integrity & Key Validation
In an enterprise dimensional data model, fact table foreign keys must have 100% coverage in dimension primary key tables (no orphan records).

In [2]:
# Check Foreign Key mappings
customer_missing = ~fact_sales['Customer ID'].isin(dim_customer['Customer ID'])
product_missing = ~fact_sales['Product ID'].isin(dim_product['Product ID'])
geo_missing = ~fact_sales['Postal Code'].isin(dim_geography['Postal Code'])
date_missing = ~fact_sales['OrderDateKey'].isin(dim_date['DateKey'])

integrity_df = pd.DataFrame([
    {"Relationship": "FactSales[Customer ID] -> DimCustomer[Customer ID]", "Orphans": int(customer_missing.sum()), "Status": "Valid"},
    {"Relationship": "FactSales[Product ID] -> DimProduct[Product ID]", "Orphans": int(product_missing.sum()), "Status": "Valid"},
    {"Relationship": "FactSales[Postal Code] -> DimGeography[Postal Code]", "Orphans": int(geo_missing.sum()), "Status": "Valid"},
    {"Relationship": "FactSales[OrderDateKey] -> DimDate[DateKey]", "Orphans": int(date_missing.sum()), "Status": "Valid"}
])
print(integrity_df.to_string(index=False))

                                       Relationship  Orphans Status
 FactSales[Customer ID] -> DimCustomer[Customer ID]        0  Valid
    FactSales[Product ID] -> DimProduct[Product ID]        0  Valid
FactSales[Postal Code] -> DimGeography[Postal Code]        0  Valid
        FactSales[OrderDateKey] -> DimDate[DateKey]        0  Valid


## 2. Power BI DAX Measures Simulation in Python
We simulate the exact formulas implemented in `powerbi/superstore_measures.dax`:
- `Total Sales = SUM(FactSales[Sales])`
- `Total Profit = SUM(FactSales[Profit])`
- `Total Quantity = SUM(FactSales[Quantity])`
- `Total Orders = DISTINCTCOUNT(FactSales[Order ID])`
- `Profit Margin % = DIVIDE([Total Profit], [Total Sales], 0)`
- `Sales Previous Year = CALCULATE([Total Sales], DATEADD(DimDate[Date], -1, YEAR))`
- `YoY Sales Growth % = DIVIDE([Total Sales] - [Sales Previous Year], [Sales Previous Year])`

In [3]:
# Base measures
total_sales = fact_sales['Sales'].sum()
total_profit = fact_sales['Profit'].sum()
total_quantity = fact_sales['Quantity'].sum()
total_orders = fact_sales['Order ID'].nunique()
profit_margin = total_profit / total_sales

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Total Quantity: {total_quantity:,}")
print(f"Total Orders: {total_orders:,}")
print(f"Profit Margin: {profit_margin * 100:.2f}%")

Total Sales: $2,297,200.86
Total Profit: $286,397.02
Total Quantity: 37,873
Total Orders: 5,009
Profit Margin: 12.47%


## 3. Time-Intelligence: Annual YoY Growth Reconciliation
Using the continuous `DimDate` calendar table, we aggregate performance across fiscal years to calculate historical YoY growth.

In [4]:
# Merge Fact with DimDate on OrderDateKey
fact_with_date = fact_sales.merge(dim_date, left_on='OrderDateKey', right_on='DateKey', how='inner')

annual_summary = fact_with_date.groupby('Year').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Total_Orders=('Order ID', 'nunique')
).reset_index()

annual_summary['Sales_Prior_Year'] = annual_summary['Total_Sales'].shift(1)
annual_summary['YoY_Sales_Growth_Pct'] = (
    (annual_summary['Total_Sales'] - annual_summary['Sales_Prior_Year']) / annual_summary['Sales_Prior_Year'] * 100
).round(2)
annual_summary['Profit_Margin_Pct'] = (annual_summary['Total_Profit'] / annual_summary['Total_Sales'] * 100).round(2)

print("--- Annual Performance & Year-Over-Year Sales Growth ---")
print(annual_summary.to_string(index=False))

--- Annual Performance & Year-Over-Year Sales Growth ---


 Year  Total_Sales  Total_Profit  Total_Orders  Sales_Prior_Year  YoY_Sales_Growth_Pct  Profit_Margin_Pct
 2015  484247.4981    49543.9741           969               NaN                   NaN              10.23
 2016  470532.5090    61618.6037          1038       484247.4981                 -2.83              13.10
 2017  609205.5980    81795.1743          1315       470532.5090                 29.47              13.43
 2018  733215.2552    93439.2696          1687       609205.5980                 20.36              12.74


## 4. Multi-Dimensional Performance Reconciliation
Verifying that dimensional slicers across Category, Region, and Segment yield exact reconciliations.

In [5]:
# Merge with product and geography dimensions
fact_full = fact_sales.merge(dim_product, on='Product ID', how='inner') \
                      .merge(dim_geography, on='Postal Code', how='inner') \
                      .merge(dim_customer, on='Customer ID', how='inner')

pivot_cat_region = fact_full.pivot_table(
    index='Category',
    columns='Region',
    values='Sales',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).round(2)

print("--- Sales Reconciliation Matrix: Category x Region ---")
print(pivot_cat_region)

--- Sales Reconciliation Matrix: Category x Region ---
Region             Central       East      South       West       Total
Category                                                               
Furniture        163797.16  208291.20  117298.68  252612.74   741999.80
Office Supplies  167026.42  205516.06  125651.31  220853.25   719047.03
Technology       170416.31  264973.98  148771.91  251991.83   836154.03
Total            501239.89  678781.24  391721.90  725457.82  2297200.86


## 5. Model Architecture Summary for Power BI Implementation
- **Fact Table**: `FactSales` (Grain: Order Line Item).
- **Dimension Tables**: `DimDate`, `DimCustomer`, `DimProduct`, `DimGeography`.
- **Relationship Type**: 1-to-Many ($1:*$), Single Direction Filter (`Dim` -> `Fact`).
- **Referential Integrity**: 100% verified with zero orphan records.
- **Next Step**: Load CSVs into Power BI Desktop, define relationships, and copy DAX expressions from `powerbi/superstore_measures.dax`.